In [1]:
## Set up

import os
import shutil
import sqlite3
import time

import pandas as pd
import matplotlib.pyplot as plt

SOURCE = r"C:\Users\shana\Downloads\DATA501\ride-knox-analysis\sql\ride_knox_v2.db" # PART A -- Built the database in SQL
WORK   = "ride_knox_work.db"

if os.path.exists(WORK):
    os.remove(WORK)
shutil.copy(SOURCE, WORK)

con = sqlite3.connect(WORK)
con.isolation_level = None     # BEGIN / COMMIT / ROLLBACK behave as written


def run(sql):
    """Execute ONE statement. Returns rows affected (-1 if it changes no rows)."""
    return con.execute(sql).rowcount


def try_run(sql):
    """For statements you EXPECT to be refused: prints the database's
    complaint instead of stopping the notebook."""
    try:
        n = con.execute(sql).rowcount
        return f"ACCEPTED ({n} rows) — is that what you expected?"
    except sqlite3.Error as e:
        return f"REFUSED -> {e}"


def timed(sql, params=(), repeat=5):
    """Run a query a few times; return the fastest run, in milliseconds."""
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        con.execute(sql, params).fetchall()
        best = min(best, time.perf_counter() - t0)
    return round(best * 1000, 1)


def db_size_mb():
    return round(os.path.getsize(WORK) / 1_048_576, 1)


print("fresh working copy:", WORK, "|", db_size_mb(), "MB")

fresh working copy: ride_knox_work.db | 36.9 MB


In [2]:
### PART A
    #Demonstrated rejections:
try_run("""
INSERT INTO stations (latitude)
VALUES ('Gay St')
""")
    # 'REFUSED -> NOT NULL constraint failed: stations.station_id'
try_run("""
INSERT INTO trips (start_station_id)
VALUES (S40)
""")
    # 'REFUSED -> no such column: S40'
try_run("""
INSERT INTO dock_changes (docks_after)
VALUES (0)
""")
    #'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

'REFUSED -> NOT NULL constraint failed: dock_changes.station_id'

In [3]:
### PART B

stations = pd.read_excel("stations_2026.xlsx")
trips25_original = pd.read_csv("trips_2025.csv")
trips26_original = pd.read_csv("trips_2026_h1.csv")

stations.to_sql("stations_sql", con, if_exists="replace", index=False)
trips25 = pd.read_sql("""SELECT * FROM trips_2025""",con)
trips26 = pd.read_sql("""SELECT * FROM trips_2026_h1""",con)

print(len(stations))
print(len(trips25))
print(len(trips26))

con.execute("BEGIN TRANSACTION;")
normalized_rider_types = con.execute("""
UPDATE trips
SET rider_type = LOWER(rider_type)
WHERE rider_type <> LOWER(rider_type);
""")
print("Number of rows affected is", normalized_rider_types.rowcount)

25
250600
137341
Number of rows affected is 0


REFLECTION AND AI DISCLOSURE

REFLECTION:

AI Disclosure: I used a generative AI tool to explain how to save the work I did in SQL, as I was having trouble saving it as anything but a SQBPRO file. I also used it to help troubleshoot why I couldn't import the SOURCE, which is because it was saved in my folder called sql/.